In [ ]:
%%sql -r dataframe_1
USE DATABASE EYPROJECT;
USE SCHEMA PUBLIC;

In [ ]:
req_text = """
pystac-client
planetary-computer
odc-stac
rasterio
xarray
rioxarray
geopandas
shapely
pyproj
"""

with open("/tmp/requirements_elevation.txt", "w") as f:
    f.write(req_text.strip())

print("Saved /tmp/requirements_elevation.txt")

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
!pip install uv
!uv pip install -r /tmp/requirements_elevation.txt

In [ ]:
import pystac_client
import planetary_computer as pc
import rasterio
from rasterio.windows import Window
import pandas as pd
import numpy as np
from tqdm import tqdm

print("All key packages imported successfully.")

In [ ]:
import pystac_client
import planetary_computer as pc
import rasterio
from odc.stac import stac_load
import pandas as pd
import numpy as np
from tqdm import tqdm
print("All key packages imported successfully.")

In [ ]:
train_df = pd.read_csv("water_quality_training_dataset.csv")
val_df = pd.read_csv("submission_template.csv")

for df in [train_df, val_df]:
    df["Sample Date"] = pd.to_datetime(
        df["Sample Date"],
        dayfirst=True,
        format="mixed",
        errors="coerce"
    ).dt.normalize()

    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce").round(4)
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce").round(4)

train_coords = train_df[["Latitude", "Longitude"]].drop_duplicates().reset_index(drop=True)
val_coords = val_df[["Latitude", "Longitude"]].drop_duplicates().reset_index(drop=True)

all_coords = pd.concat([train_coords, val_coords], axis=0).drop_duplicates().reset_index(drop=True)

print("train_df rows:", len(train_df))
print("val_df rows:", len(val_df))
print("unique training coords:", len(train_coords))
print("unique validation coords:", len(val_coords))
print("total unique coords to extract:", len(all_coords))
display(all_coords.head())

In [ ]:
for df in [train_df, val_df]:
    df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce").round(4)
    df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce").round(4)

In [ ]:
def get_dem_dataset(lat, lon):
    try:
        search = catalog.search(
            collections=[collection],
            intersects={
                "type": "Point",
                "coordinates": [float(lon), float(lat)]
            }
        )

        items = list(search.items())
        if len(items) == 0:
            return None

        item = items[0]
        href = pc.sign(item.assets["data"].href)
        return rasterio.open(href)

    except Exception as e:
        print(f"DEM search failed for lat={lat}, lon={lon}: {e}")
        return None

In [ ]:
def extract_terrain_features(lat, lon, window_size=5):
    src = get_dem_dataset(lat, lon)

    if src is None:
        return {
            "elevation": np.nan,
            "slope_deg": np.nan,
            "local_relief": np.nan,
            "roughness": np.nan
        }

    try:
        row, col = src.index(float(lon), float(lat))

        half = window_size // 2
        win = Window(col - half, row - half, window_size, window_size)

        arr = src.read(1, window=win, masked=True).astype("float64")

        if arr.size == 0 or np.all(arr.mask):
            src.close()
            return {
                "elevation": np.nan,
                "slope_deg": np.nan,
                "local_relief": np.nan,
                "roughness": np.nan
            }

        arr_filled = np.where(arr.mask, np.nan, arr)

        center_val = arr_filled[half, half]
        local_relief = np.nanmax(arr_filled) - np.nanmin(arr_filled)
        roughness = np.nanstd(arr_filled)

        xres = abs(src.transform.a)
        yres = abs(src.transform.e)

        gy, gx = np.gradient(arr_filled, yres, xres)
        slope_rad = np.arctan(np.sqrt(gx**2 + gy**2))
        slope_deg = np.degrees(np.nanmean(slope_rad))

        src.close()

        return {
            "elevation": float(center_val) if not np.isnan(center_val) else np.nan,
            "slope_deg": float(slope_deg) if not np.isnan(slope_deg) else np.nan,
            "local_relief": float(local_relief) if not np.isnan(local_relief) else np.nan,
            "roughness": float(roughness) if not np.isnan(roughness) else np.nan
        }

    except Exception as e:
        try:
            src.close()
        except:
            pass
        print(f"Terrain extraction failed for lat={lat}, lon={lon}: {e}")
        return {
            "elevation": np.nan,
            "slope_deg": np.nan,
            "local_relief": np.nan,
            "roughness": np.nan
        }

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

collection = "cop-dem-glo-90"

In [ ]:
terrain_rows = []

for i, (_, row) in enumerate(tqdm(all_coords.iterrows(), total=len(all_coords))):
    lat = row["Latitude"]
    lon = row["Longitude"]

    feats = extract_terrain_features(lat, lon, window_size=5)

    terrain_rows.append({
        "Latitude": lat,
        "Longitude": lon,
        **feats
    })

    if (i + 1) % 100 == 0:
        checkpoint = pd.DataFrame(terrain_rows)
        checkpoint.to_csv("/tmp/terrain_lookup_checkpoint.csv", index=False)
        print(f"Saved checkpoint at {i+1} rows")

terrain_lookup = pd.DataFrame(terrain_rows).drop_duplicates(subset=["Latitude", "Longitude"])

print("Expected unique coords:", len(all_coords))
print("Terrain lookup shape:", terrain_lookup.shape)
print("Unique terrain coords:", terrain_lookup[['Latitude', 'Longitude']].drop_duplicates().shape[0])

display(terrain_lookup.head())

print("Missing terrain values in lookup:")
print(terrain_lookup[["elevation", "slope_deg", "local_relief", "roughness"]].isna().sum())

if len(terrain_lookup) < 0.95 * len(all_coords):
    raise ValueError(
        f"Terrain extraction incomplete: expected about {len(all_coords)} rows, got {len(terrain_lookup)}"
    )

In [ ]:
train_terrain = train_df.merge(
    terrain_lookup,
    on=["Latitude", "Longitude"],
    how="left"
)

val_terrain = val_df.merge(
    terrain_lookup,
    on=["Latitude", "Longitude"],
    how="left"
)

train_terrain = train_terrain[
    ["Latitude", "Longitude", "Sample Date", "elevation", "slope_deg", "local_relief", "roughness"]
]

val_terrain = val_terrain[
    ["Latitude", "Longitude", "Sample Date", "elevation", "slope_deg", "local_relief", "roughness"]
]

print("train_terrain shape:", train_terrain.shape)
print("val_terrain shape:", val_terrain.shape)

print("Expected train rows:", len(train_df))
print("Expected val rows:", len(val_df))

print("Missing train terrain rows:")
print(train_terrain[["elevation", "slope_deg", "local_relief", "roughness"]].isna().sum())

print("Missing val terrain rows:")
print(val_terrain[["elevation", "slope_deg", "local_relief", "roughness"]].isna().sum())

In [ ]:
train_terrain.to_csv("/tmp/terrain_features_training.csv", index=False)
val_terrain.to_csv("/tmp/terrain_features_validation.csv", index=False)

print("Saved terrain feature files.")

In [ ]:
session.sql("""
PUT file:///tmp/terrain_features_training.csv @~ AUTO_COMPRESS=FALSE OVERWRITE=TRUE
""").collect()

session.sql("""
PUT file:///tmp/terrain_features_validation.csv @~ AUTO_COMPRESS=FALSE OVERWRITE=TRUE
""").collect()

print("Uploaded terrain files to stage.")